In [1]:
import os
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

In [2]:
# Configurar dispositivo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cargar el modelo y el procesador de CLIP
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)

# Ruta a las imágenes
IMAGE_DIR = "./Data/Images"

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
def zero_shot_classification(image_path, candidate_labels):
    # Cargar la imagen
    image = Image.open(image_path).convert("RGB")

    # Procesar imagen y textos juntos
    inputs = processor(
        text=candidate_labels, images=image, return_tensors="pt", padding=True
    ).to(device)

    # Pasar por el modelo
    with torch.no_grad():
        outputs = model(**inputs)

    # CLIP calcula los logits de similitud. Aplicamos Softmax para obtener probabilidades.
    logits_per_image = outputs.logits_per_image  # Similitud imagen-texto
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()[0]

    # Mostrar resultados ordenados
    results = sorted(zip(candidate_labels, probs), key=lambda x: x[1], reverse=True)
    return results

In [4]:
# Ejemplo de uso:
image_test = os.path.join(IMAGE_DIR, "1000268201_693b08cb0e.jpg")
labels = ["a dog playing in the grass", "a child running", "a car on the street", "a person riding a bike"]
print(zero_shot_classification(image_test, labels))

[('a child running', np.float32(0.99704856)), ('a person riding a bike', np.float32(0.002249011)), ('a car on the street', np.float32(0.0004405091)), ('a dog playing in the grass', np.float32(0.00026199326))]


In [5]:
# Cargar el dataset (ajusta la ruta si es necesario)
df = pd.read_csv("./Data/captions.txt")  # Estructura típica: image, caption

# Obtener una lista de imágenes únicas (Flickr8k tiene 5 subtítulos por imagen)
unique_images = df["image"].unique().tolist()
all_captions = df["caption"].tolist()

In [6]:
image_embeddings = []
valid_image_paths = []

print("Extrayendo embeddings de imágenes...")
for img_name in unique_images[:500]:  # Limitado a 500 para el ejemplo
    img_path = os.path.join(IMAGE_DIR, img_name)
    if os.path.exists(img_path):
        try:
            image = Image.open(img_path).convert("RGB")
            inputs = processor(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                img_emb = model.get_image_features(**inputs)
                # Normalizar el vector para que el producto punto sea igual a la similitud coseno
                img_emb = img_emb / img_emb.norm(p=2, dim=-1, keepdim=True)
                image_embeddings.append(img_emb)
                valid_image_paths.append(img_path)
        except Exception as e:
            continue

# Concatenar todos los embeddings en un solo tensor
image_embeddings = torch.cat(image_embeddings, dim=0)

Extrayendo embeddings de imágenes...


In [7]:
print("Extrayendo embeddings de texto...")
# Limitado a los primeros 1000 textos para el ejemplo
sample_captions = all_captions[:1000]

inputs_text = processor(
    text=sample_captions, return_tensors="pt", padding=True, truncation=True
).to(device)
with torch.no_grad():
    text_embeddings = model.get_text_features(**inputs_text)
    text_embeddings = text_embeddings / text_embeddings.norm(p=2, dim=-1, keepdim=True)

Extrayendo embeddings de texto...


In [8]:
def text_to_image_retrieval(query_text, top_k=3):
    # 1. Obtener embedding del texto de consulta
    inputs = processor(text=[query_text], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        query_emb = model.get_text_features(**inputs)
        query_emb = query_emb / query_emb.norm(p=2, dim=-1, keepdim=True)

    # 2. Calcular similitud (producto punto de vectores normalizados)
    similarities = (query_emb @ image_embeddings.T).squeeze(0)

    # 3. Obtener los mejores resultados
    top_indices = torch.topk(similarities, k=top_k).indices.cpu().numpy()

    print(f"\nResultados para la búsqueda: '{query_text}'")
    for idx in top_indices:
        print(
            f"- Imagen: {valid_image_paths[idx]} (Similitud: {similarities[idx].item():.4f})"
        )

In [9]:
# Ejemplo de uso:
text_to_image_retrieval("a black dog catching a frisbee", top_k=3)


Resultados para la búsqueda: 'a black dog catching a frisbee'
- Imagen: ./Data/Images/1026685415_0431cbf574.jpg (Similitud: 0.3466)
- Imagen: ./Data/Images/1436760519_8d6101a0ed.jpg (Similitud: 0.3329)
- Imagen: ./Data/Images/1067180831_a59dc64344.jpg (Similitud: 0.3223)


In [13]:
def image_to_text_retrieval(image_path, top_k=10):
    # 1. Obtener embedding de la imagen de consulta
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        query_emb = model.get_image_features(**inputs)
        query_emb = query_emb / query_emb.norm(p=2, dim=-1, keepdim=True)

    # 2. Calcular similitud con todos los textos indexados
    similarities = (query_emb @ text_embeddings.T).squeeze(0)

    # 3. Obtener los mejores resultados
    top_indices = torch.topk(similarities, k=top_k).indices.cpu().numpy()

    print(f"\nResultados de texto para la imagen: {image_path}")
    for idx in top_indices:
        print(
            f"- Descripción: '{sample_captions[idx]}' (Similitud: {similarities[idx].item():.4f})"
        )




In [16]:
test_img = os.path.join(IMAGE_DIR, "1000268201_693b08cb0e.jpg")
image_to_text_retrieval(test_img, top_k=20)


Resultados de texto para la imagen: ./Data/Images/1000268201_693b08cb0e.jpg
- Descripción: 'A little girl in a pink dress going into a wooden cabin .' (Similitud: 0.3460)
- Descripción: 'A little girl climbing the stairs to her playhouse .' (Similitud: 0.3456)
- Descripción: 'A little girl climbing into a wooden playhouse .' (Similitud: 0.3392)
- Descripción: 'A child in a pink dress is climbing up a set of stairs in an entry way .' (Similitud: 0.3308)
- Descripción: 'A girl going into a wooden building .' (Similitud: 0.2973)
- Descripción: 'A lady holds a little girl who is trying to catch bubbles .' (Similitud: 0.2783)
- Descripción: 'A woman is holding a little girl who is trying to catch bubbles .' (Similitud: 0.2747)
- Descripción: 'A child is standing on her head .' (Similitud: 0.2679)
- Descripción: 'A little girl standing on her head .' (Similitud: 0.2628)
- Descripción: 'The little boy is wearing a black pinstripe shirt and walking on a deck .' (Similitud: 0.2613)
- Descripci